The following will install the full Modular platform suite, including all dependencies for serving models. If you just want to work with the MAX framework for GPU programming, replace `modular` in the below with `max` and the installation will be slightly faster.

In [1]:
!pip install modular --index-url https://dl.modular.com/public/nightly/python/simple/ --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://dl.modular.com/public/nightly/python/simple/, https://download.pytorch.org/whl/cpu


First, verify that we're running on a GPU-enabled instance. The first part of this tutorial will run fine on an NVIDIA T4 GPU, available in the free tier of Google Colab. An L4 or A100 GPU from the Colab Pro is necessary to run the full LLM example at the end of this notebook.

In [2]:
!nvidia-smi

Tue May  6 16:10:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Set up the appropriate imports from the MAX libraries.

In [3]:
import numpy as np
from max.driver import CPU, Accelerator, Tensor, accelerator_count
from max.dtype import DType
from max.engine import InferenceSession
from max.graph import DeviceRef, Graph, TensorType, ops

We'll verify that MAX can find a usable accelerator. If the below count is 0, make sure that you're running on a GPU-enabled Colab instance.

In [4]:
accelerator_count()

1

Next, we'll get an accelerator device, if one is available:

In [5]:
device = CPU() if accelerator_count() == 0 else Accelerator()
device

Device(type=gpu,id=0)

Now, we'll configure a computational graph to run on the GPU. This is a very simple, one-operation graph that takes in two vectors, adds them in parallel, and returns the result.

In [6]:
vector_width = 10
dtype = DType.float32

with Graph(
    "vector_addition",
    input_types=[
        TensorType(
            dtype,
            shape=[vector_width],
            device=DeviceRef.from_device(device),
        ),
        TensorType(
            dtype,
            shape=[vector_width],
            device=DeviceRef.from_device(device),
        ),
    ],
) as graph:
    lhs, rhs = graph.inputs
    output = lhs + rhs
    graph.output(output)

Once the graph is configured, we can compile it for our accelerator:

In [7]:
session = InferenceSession(
    devices=[device],
)

model = session.load(graph)

MAX provides transparent interoperability with NumPy arrays, so we can initialize a couple of random vectors with NumPy and create input Tensors from them:

In [8]:
lhs_values = np.random.uniform(size=(vector_width)).astype(np.float32)
rhs_values = np.random.uniform(size=(vector_width)).astype(np.float32)

lhs_tensor = Tensor.from_numpy(lhs_values).to(device)
rhs_tensor = Tensor.from_numpy(rhs_values).to(device)

Then we can run the graph on the GPU and copy values back to the host:

In [9]:
result = model.execute(lhs_tensor, rhs_tensor)[0]

result = result.to(CPU())

Finally, we can print the inputs and results, and compare to a similar calculation in NumPy:

In [10]:
print("Left-hand-side values:")
print(lhs_values)
print()

print("Right-hand-side values:")
print(rhs_values)
print()

print("Graph result:")
print(result.to_numpy())
print()

print("Expected result:")
print(lhs_values + rhs_values)

Left-hand-side values:
[0.15611424 0.27276358 0.35734496 0.35179263 0.6384797  0.8303689
 0.66703176 0.01309488 0.5253851  0.02661598]

Right-hand-side values:
[0.13239917 0.9140202  0.00356107 0.48741138 0.34380272 0.18117483
 0.13447495 0.5396943  0.54703164 0.31383783]

Graph result:
[0.28851342 1.1867838  0.36090603 0.839204   0.9822824  1.0115438
 0.8015067  0.5527892  1.0724168  0.3404538 ]

Expected result:
[0.28851342 1.1867838  0.36090603 0.839204   0.9822824  1.0115438
 0.8015067  0.5527892  1.0724168  0.3404538 ]


After demonstrating a very basic graph, we can try out a full LLM running and generating text from a prompt. The following will only run on a higher-end GPU in Colab, such as an L4 or A100.

In [11]:
from max.entrypoints.llm import LLM
from max.pipelines import PipelineConfig
from max.serve.config import Settings

The following code loads an LLM (Qwen 2.5, 0.5B parameters), provides it with a prompt, and then collects the output. One a first run, the full LLM graph will be compiled and optimized for the target GPU, and subsequent runs will be much faster with the compiled graph.

In [12]:
model_path = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading model: {model_path}")
pipeline_config = PipelineConfig(model_path=model_path)
settings = Settings()
llm = LLM(settings, pipeline_config)

prompts = [
    "The fastest way to learn python is",
]

print("Generating responses...")
responses = llm.generate(prompts, max_new_tokens=50)

for i, (prompt, response) in enumerate(zip(prompts, responses)):
    print(f"========== Response {i} ==========")
    print(prompt + response)
    print()

Loading model: Qwen/Qwen2.5-0.5B-Instruct


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generating responses...


100%|██████████| 1/1 [00:00<00:00,  1.97it/s]

========== Response 0 ==========
The fastest way to learn python is to practice coding. The best way to learn python is to read python books. The best way to learn python is to read python books. The best way to learn python is to read python books. The best way to learn python is to read python

